In [1]:
import os
import shutil

import pandas as pd
import yaml

Example of project yaml:
```yaml
# Ultralytics YOLO 🚀, AGPL-3.0 license
# COCO128 dataset https://www.kaggle.com/ultralytics/coco128 (first 128 images from COCO train2017) by Ultralytics
# Example usage: yolo train data=coco128.yaml
# parent
# ├── ultralytics
# └── datasets
#     └── coco128  ← downloads here (7 MB)


# Train/val/test sets as 1) dir: path/to/imgs, 2) file: path/to/imgs.txt, or 3) list: [path/to/imgs1, path/to/imgs2, ..]
path: ../datasets/position_db_pose  # dataset root dir
train: images/train  # train images (relative to 'path') 128 images
val: images/val  # val images (relative to 'path') 128 images
test: images/test  # test images (optional)

kpt_shape: [1, 2]

# Classes
names:
  0: person
```

Example of annotaion txt file
```txt
0 0.726953125 0.3953125 0.27265625 0.20104166666666667 0.63359375 0.453125
0 0.288671875 0.35885416666666664 0.16484375 0.17395833333333333 0.36015625 0.41458333333333336
0 0.507421875 0.1359375 0.10703125 0.271875 0.49296875 0.25104166666666666
0 0.59765625 0.07864583333333333 0.090625 0.15729166666666666 0.58203125 0.10833333333333334
```

In [2]:
base_path = "../../../db/object-detection"
df = pd.read_csv(os.path.join(base_path, "annotation.csv"))
df.head()

,label,occluded,truncated,not_ground,occupied,occupied_by_person,ID,xtl,ytl,xbr,ybr,x,y,img_id,id
0,meeting_room_table,True,False,False,False,False,8,51.60,186.10,307.89,332.61,172.16,260.70,8e945fd6-8327-42f6-bfd1-b828a4a82dff,23bef311-d40e-44c3-8dee-bf24118ea02a
1,bag,False,False,False,False,False,9,346.45,165.42,393.28,220.25,369.91,191.48,8e945fd6-8327-42f6-bfd1-b828a4a82dff,ece73ea1-9859-4c7c-a843-30a4c68762e6
2,bag,True,False,False,False,False,10,329.10,133.89,369.63,170.60,350.15,150.37,8e945fd6-8327-42f6-bfd1-b828a4a82dff,e429a150-1202-42a7-9183-c1e22b9fe01c
3,laptop_closed,False,False,False,False,False,19,224.38,287.20,280.56,324.61,253.60,305.88,8e945fd6-8327-42f6-bfd1-b828a4a82dff,34fe393f-b796-45c1-8d9d-b8f2372d6726
4,laptop_opened,False,False,False,False,False,18,197.42,191.27,249.80,228.00,224.38,210.00,8e945fd6-8327-42f6-bfd1-b828a4a82dff,db41ee80-1bd3-4b93-9178-f152342ad08a


Algo:
1. Read the annotaion and images;
2. Filter: remove: images without annotations, with errors, duplicates;
3. Select specific classes, merge if needed;
4. Map labels to class ids;
5. Convert to list of dict:
```python
{
    "id",
    "path",
    "objects": {
        "id",
        "label",
        "bbox",
        "position"
    }
}
```
6. Create YOLO dataset:
    1. label txt
    2. copy images
    3. create project.yaml file

In [27]:
def create_project_file(file_pth: str, dataset_pth: str, label_mapping: dict) -> None:
    data = {
        "path": dataset_pth,
        "train": "images/train",
        "val": "images/val",
        "kpt_shape": [1, 2],
    }

    with open(file_pth, "wt") as f:
        yaml.dump(data, f, sort_keys=False, default_flow_style=None)
    
    with open(file_pth, "at") as f:
        yaml.dump({"names": label_mapping}, f, sort_keys=False)

In [28]:
create_project_file("tst.yaml", "tst", {0: "desk"})